# Kaggle：最终模型 terrain-only 快速筛选（seed1337）

从匹配seed的完整 A′→CMCR checkpoint 开始，冻结ResNet50编码器和BatchNorm，使用CE+0.2 Lovasz适配四个地形通道。只使用Val选模，不访问Test。

In [ ]:
from pathlib import Path
import hashlib, importlib.metadata, importlib.util, json, os, subprocess, sys

REPO_URL = 'https://github.com/song110585-cpu/lunar-linear.git'
REPO_BRANCH = 'test-new-module'
REQUIRED_COMMIT = 'e015290'
REPO_DIR = Path('/kaggle/working/lunar-linear')
PROJECT_DIR = REPO_DIR / 'LTL-Net'
OUTPUT_ROOT = Path('/kaggle/working')
CONFIG_NAME = 'v6_overlap40_joint_finetune_rezero_cmcr_lovasz02_terrain_only_seed1337.json'
EXPECTED_SEED = 1337
EXPECTED_CHECKPOINT_SHA256 = '012728c1f4deb888eff474fd49d6ea1478fb7803f61348bce8f3b0c2e79e3c04'

DATA_CANDIDATES = [
    Path('/kaggle/input/datasets/yuanssy/datav6-overlap40/dataset_v6_random811_overlap40'),
    Path('/kaggle/input/datasets/changyasong/datav6-overlap40/dataset_v6_random811_overlap40'),
    Path('/kaggle/input/datasets/changyasong/v6data/dataset_v6_random811_overlap40'),
    Path('/kaggle/input/datav6-overlap40/dataset_v6_random811_overlap40'),
    Path('/kaggle/input/v6data/dataset_v6_random811_overlap40'),
]

In [ ]:
required = [('rasterio', 'rasterio'), ('tqdm', 'tqdm')]
missing = [package for module, package in required if importlib.util.find_spec(module) is None]
try:
    smp_version = importlib.metadata.version('segmentation-models-pytorch')
except importlib.metadata.PackageNotFoundError:
    smp_version = None
if smp_version != '0.5.0':
    missing.append('segmentation-models-pytorch==0.5.0')
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])

if not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)])
elif not (REPO_DIR / '.git').is_dir():
    raise RuntimeError(f'目录存在但不是Git仓库: {REPO_DIR}')
else:
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH])
subprocess.check_call(['git', '-C', str(REPO_DIR), 'merge-base', '--is-ancestor', REQUIRED_COMMIT, 'HEAD'])
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('Git commit:', commit)
subprocess.run(['nvidia-smi'], check=False)

In [ ]:
discovered_data = sorted({
    path for path in Path('/kaggle/input').rglob('dataset_v6_random811_overlap40')
    if path.is_dir()
})
ordered_data = []
for path in [*DATA_CANDIDATES, *discovered_data]:
    if path not in ordered_data and (path / 'dataset_protocol.json').is_file():
        ordered_data.append(path)
assert ordered_data, '未找到dataset_v6_random811_overlap40；请先通过Add Input挂载数据集'
DATA_ROOT = ordered_data[0]

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

checkpoint_candidates = sorted(Path('/kaggle/input').rglob('*.pth'))
checkpoint_matches = [
    path for path in checkpoint_candidates
    if path.is_file() and sha256_file(path) == EXPECTED_CHECKPOINT_SHA256
]
assert len(checkpoint_matches) == 1, (
    f'应恰好挂载一个seed{EXPECTED_SEED}初始权重，实际匹配={checkpoint_matches}；'
    f'期望SHA-256={EXPECTED_CHECKPOINT_SHA256}'
)
INIT_CHECKPOINT = checkpoint_matches[0]
CONFIG_PATH = PROJECT_DIR / 'configs' / CONFIG_NAME
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
assert config['seed'] == EXPECTED_SEED
assert config['channel_mode'] == 'terrain_only' and config['lovasz_weight'] == 0.2
assert config['joint_finetune'] is True and config['automatic_test_evaluation'] is False
assert config['expected_init_checkpoint_sha256'] == EXPECTED_CHECKPOINT_SHA256
print('数据目录:', DATA_ROOT)
print('初始权重:', INIT_CHECKPOINT)
print('实验:', config['run_name'])
print('输出:', OUTPUT_ROOT / f"result_{config['run_name']}")

In [ ]:
command = [
    sys.executable, str(PROJECT_DIR / 'scripts/run_autodl_joint_finetune.py'),
    '--project-dir', str(PROJECT_DIR),
    '--config', str(CONFIG_PATH),
    '--data-dir', str(DATA_ROOT),
    '--output-dir', str(OUTPUT_ROOT),
    '--init-checkpoint', str(INIT_CHECKPOINT),
]
env = os.environ.copy(); env['PYTHONUNBUFFERED'] = '1'
print(' '.join(command), flush=True)
subprocess.check_call(command, cwd=PROJECT_DIR, env=env)

In [ ]:
result_dir = OUTPUT_ROOT / f"result_{config['run_name']}"
metrics_path = result_dir / 'metrics.json'
archive_path = Path(str(result_dir) + '.zip')
assert metrics_path.is_file(), metrics_path
assert archive_path.is_file(), archive_path
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
assert metrics['channel_mode'] == 'terrain_only' and metrics['test_evaluated'] is False
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print('下载:', archive_path)